In [1]:
# IMPORTS
import torch
import torch.nn as nn
from Binn import BINN
import data_handling as dh
import binn_training as bt
import os 
import custom_train_test_split as ctts
import pandas as pd, numpy as np

# Supress warnings
import warnings
warnings.filterwarnings("ignore")

In [2]:
# GLOBALS
ALL_CELLTYPES = [0,1,2,3,4,5,6,7,8]
EPOCHS = 20
TRAIN_SIZE = 0.8
BATCH_SIZE = 16
MASK_PATHS = [f"/data/shared/alzgene26/PathwayData/MaskMatrixLayers/mg_200_mc_200_mhvg1000/oligo_exc3_exc2_vasc_immune_astro_inhi_opcs_exc1_layer_{i}_mask.csv" 
              for i in range(5)]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_path = "/data/shared/alzgene26"
data_path = base_path + "/data/processed_data/completed/mg_200_mc_200_mhvg1000/"
comp_proc_data_path = base_path + "/data/processed_data/extracted_from_completed/"
comp_proc_data_path

'/data/shared/alzgene26/data/processed_data/extracted_from_completed/'

In [3]:
print("Reading completed processed adata...")
datasets = ctts.read_files(to_include=ALL_CELLTYPES, filepath=comp_proc_data_path)

Reading completed processed adata...
Labels to include: ['astro', 'exc1', 'exc2', 'exc3', 'immune', 'inhi', 'oligo', 'opcs', 'vasc']
Reading astro
Reading exc1
Reading exc2
Reading exc3
Reading immune
Reading inhi
Reading oligo
Reading opcs
Reading vasc


In [4]:
masks = dh.read_masks(MASK_PATHS, print_shapes=True)

Matrix 0 shape: (8075, 1160)
Matrix 1 shape: (1160, 551)
Matrix 2 shape: (551, 188)
Matrix 3 shape: (188, 29)
Matrix 4 shape: (29, 1)


In [5]:
print("Aligning adatas to BINN...")
datasets_aligend = dh.subset_genes(datasets, masks['df0'])

Aligning adatas to BINN...


In [6]:
print("Padding adatas to BINN-ready shape...")
datasets_padded = dh.pad_align_data(datasets_aligend, masks["df0"])
datasets_padded

Padding adatas to BINN-ready shape...


{'astro': AnnData object with n_obs × n_vars = 1234 × 8075
     obs: 'subject', 'cell_type_high_resolution', 'n_obs_aggregated', 'AD_status',
 'exc1': AnnData object with n_obs × n_vars = 426 × 8075
     obs: 'subject', 'cell_type_high_resolution', 'n_obs_aggregated', 'AD_status',
 'exc2': AnnData object with n_obs × n_vars = 1661 × 8075
     obs: 'subject', 'cell_type_high_resolution', 'n_obs_aggregated', 'AD_status',
 'exc3': AnnData object with n_obs × n_vars = 3629 × 8075
     obs: 'subject', 'cell_type_high_resolution', 'n_obs_aggregated', 'AD_status',
 'immune': AnnData object with n_obs × n_vars = 1645 × 8075
     obs: 'subject', 'cell_type_high_resolution', 'n_obs_aggregated', 'AD_status',
 'inhi': AnnData object with n_obs × n_vars = 9724 × 8075
     obs: 'subject', 'cell_type_high_resolution', 'n_obs_aggregated', 'AD_status',
 'oligo': AnnData object with n_obs × n_vars = 427 × 8075
     obs: 'subject', 'cell_type_high_resolution', 'n_obs_aggregated', 'AD_status',
 'opcs': An

In [7]:
print("Creating AnnCollection...")
acollection = ctts.create_encoded_collection(datasets_padded)
acollection
# NOTE: join_vars="inner" because "outer" would create artificial zeroes in gene expresion???

Creating AnnCollection...


AnnCollection object with n_obs × n_vars = 20724 × 8075
  constructed from 9 AnnData objects
    obs: 'subject', 'cell_type_high_res', 'n_obs_aggregated', 'AD_status', 'cell_type_low_res'

In [8]:
print("Creating train/test split...")
train_adata, test_adata = ctts.custom_train_test_split(acollection, train_size=TRAIN_SIZE)

Creating train/test split...
Train Subjects: 341
Test Subjects: 86


In [9]:
print("Getting dataloaders...")
train_loader, test_loader = dh.create_dataloaders(train_adata, test_adata)

Getting dataloaders...


In [10]:
# Extract pure number representation matrices from masks
mask_matrix_list = [masks[mask].to_numpy() for mask in masks]

# Starting amount of features
in_features = masks["df0"].shape[0]

# Extract layer dimensions
layers_list = [masks[mask].shape[1] for mask in masks]

print(f"input features: {in_features}")
print(f"layer list: {layers_list}")

input features: 8075
layer list: [1160, 551, 188, 29, 1]


In [11]:
# Conversion for mask matrix list, creates tensors for BINN, transposed
tensor_masks = [torch.tensor(mask).float() for mask in mask_matrix_list]
# Put on device for BINN
tensor_masks = [mask.to(device) for mask in tensor_masks]

In [12]:
model = BINN(in_features=in_features,
                  layers_list=layers_list,
                  mask_list=tensor_masks).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [13]:
# Step through the training
for epoch in range(5):
    train_loss, train_acc = bt.train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = bt.test_one_epoch(model, test_loader, criterion, device)
    
    print(f"Epoch {epoch+1} / {5}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Test Loss:  {test_loss:.4f} | Test Acc:  {test_acc:.4f}")
    print("-" * 30)


Epoch 1 / 5
Train Loss: 0.6915 | Train Acc: 7.2868
Test Loss:  0.6797 | Test Acc:  6.4693
------------------------------
Epoch 2 / 5
Train Loss: 0.6903 | Train Acc: 7.2868
Test Loss:  0.6794 | Test Acc:  6.4693
------------------------------
Epoch 3 / 5
Train Loss: 0.6902 | Train Acc: 7.2868
Test Loss:  0.6794 | Test Acc:  6.4693
------------------------------
Epoch 4 / 5
Train Loss: 0.6901 | Train Acc: 7.2868
Test Loss:  0.6793 | Test Acc:  6.4693
------------------------------
Epoch 5 / 5
Train Loss: 0.6899 | Train Acc: 7.2868
Test Loss:  0.6789 | Test Acc:  6.4693
------------------------------
